# 00. 원본 데이터 점검

`config/default.yaml`에 등록한 VM 원본 데이터 경로를 기준으로 메타데이터와 RFP 파일의 대응 여부를 확인합니다.

- 메타데이터 행 수와 컬럼 확인
- 결측치와 사업명 중복 확인
- 실제 PDF/HWP 파일 수 및 형식 분포 확인
- `파일명` 기준으로 CSV와 실제 원본 파일의 누락 여부 확인

In [ ]:
from pathlib import Path

import pandas as pd
import yaml

CONFIG_PATH = Path('../config/default.yaml')

with CONFIG_PATH.open(encoding='utf-8') as file:
    config = yaml.safe_load(file)

metadata_path = Path(config['paths']['metadata'])
raw_documents_path = Path(config['paths']['raw_documents'])

print(f'메타데이터 경로: {metadata_path}')
print(f'원본 문서 경로: {raw_documents_path}')

if not metadata_path.exists() or not raw_documents_path.exists():
    raise FileNotFoundError('VM에서 노트북을 실행하고 config/default.yaml의 경로를 확인하세요.')

In [ ]:
df = pd.read_csv(metadata_path)

print('=== 메타데이터 기본 정보 ===')
print(f'행 수: {len(df):,}')
print(f'컬럼: {df.columns.tolist()}')
display(df.head())

In [ ]:
print('=== 컬럼별 결측치 ===')
missing_counts = df.isna().sum()
missing_columns = missing_counts[missing_counts > 0]

if missing_columns.empty:
    print('결측치가 없습니다.')
else:
    print(missing_columns.to_string())

print('\n=== 사업명 중복 ===')
duplicate_projects = df[df.duplicated('사업명', keep=False)]

if duplicate_projects.empty:
    print('중복된 사업명이 없습니다.')
else:
    print(duplicate_projects[['사업명', '파일명']].sort_values('사업명').to_string(index=False))

In [ ]:
raw_files = sorted(
    path for path in raw_documents_path.rglob('*')
    if path.is_file() and path.suffix.lower() in {'.pdf', '.hwp'}
)

raw_file_names = {path.name for path in raw_files}

print('=== 실제 원본 파일 ===')
print(f'PDF/HWP 파일 수: {len(raw_files):,}')
print(pd.Series([path.suffix.lower() for path in raw_files]).value_counts().to_string())

In [ ]:
metadata_file_names = set(df['파일명'].dropna().astype(str).str.strip())
metadata_only = sorted(metadata_file_names - raw_file_names)
raw_only = sorted(raw_file_names - metadata_file_names)

print('=== 메타데이터-원본 파일 매칭 ===')
print(f'CSV 파일명 수: {len(metadata_file_names):,}')
print(f'실제 원본 파일명 수: {len(raw_file_names):,}')
print(f'CSV에만 있는 파일: {len(metadata_only):,}')
print(f'원본 폴더에만 있는 파일: {len(raw_only):,}')

if metadata_only:
    print('\n[CSV에만 있는 파일]')
    print('\n'.join(metadata_only))

if raw_only:
    print('\n[원본 폴더에만 있는 파일]')
    print('\n'.join(raw_only))

if not metadata_only and not raw_only:
    print('메타데이터와 원본 파일명이 모두 일치합니다.')